In [ ]:
# --- Setup: dataset onto Colab's local disk (run once per session) ---
import os, sys, time, zipfile

DATA = "/content/data"
if not os.path.isdir(DATA):
    from google.colab import drive
    drive.mount("/content/drive")
    t = time.time()
    with zipfile.ZipFile("/content/drive/MyDrive/data.zip") as z:
        z.extractall("/content")
    print(f"unzipped to {DATA} in {time.time()-t:.0f}s")

sys.path.append(f"{DATA}/scripts")
from load_dataset import Dataset

ds = Dataset(DATA)
print(f"camera: {len(ds.camera)} frames | gt: {len(ds.gt)} poses | "
      f"span: {(ds.gt.t_ns.iloc[-1] - ds.gt.t_ns.iloc[0]) / 1e9:.1f} s | "
      f"images on disk: {len(os.listdir(f'{DATA}/camera/images'))}")

# 1 · The experiment at a glance

In [ ]:
# Where: the ground-truth lap on the lidar map
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#c3c2b7", "xtick.color": "#898781",
                     "ytick.color": "#898781", "grid.color": "#e1e0d9",
                     "grid.linewidth": 0.6})

mp = np.load(f"{DATA}/ground_truth/map_points.npz")["xy"]
tg = (ds.gt.t_ns - ds.gt.t_ns.iloc[0]) / 1e9

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(*mp.T, s=0.6, c="0.3", lw=0, rasterized=True)
sc = ax.scatter(ds.gt.x, ds.gt.y, c=tg, s=5, cmap="viridis", lw=0, zorder=3)
ax.plot(ds.gt.x.iloc[0], ds.gt.y.iloc[0], "o", ms=9, mfc="none", mec="k", mew=1.5, zorder=5)
ax.annotate("start = end (loop closes)", xy=(ds.gt.x.iloc[0], ds.gt.y.iloc[0]),
            xytext=(-8, 14), textcoords="offset points", fontsize=9, ha="right")
ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_xlim(ds.gt.x.min() - 1.5, ds.gt.x.max() + 1.6)
ax.set_ylim(ds.gt.y.min() - 1.3, ds.gt.y.max() + 1.5)
fig.colorbar(sc, ax=ax, label="time [s]", shrink=0.8)
ax.set_title("One 17.8 m lap in 117 s — lidar map + ground-truth trajectory")
plt.show()

In [ ]:
# When: five async streams on one clock axis
t0 = int(ds.camera.t_ns.iloc[0])
sec = lambda s: (np.asarray(s, dtype=np.int64) - t0) / 1e9
cam_t, imu_t, odo_t, gt_t = sec(ds.camera.t_ns), sec(ds.imu.t_ns), sec(ds.odom.t_ns), sec(ds.gt.t_ns)
wf = ds.wifi.groupby("scan_idx").agg(a=("t_start_ns", "first"), b=("t_end_ns", "first"))
w0, w1 = sec(wf.a), sec(wf.b)

v = np.hypot(np.diff(ds.gt.x), np.diff(ds.gt.y)) / np.diff(gt_t)
mv = np.convolve(v, np.ones(9) / 9, "same") > 0.03          # smoothed |v| vs GT noise floor
tA, tB = gt_t[np.argmax(mv)], gt_t[len(mv) - np.argmax(mv[::-1])]

rows = [("camera  ~30 Hz", cam_t, "#2a78d6"), ("imu  20 Hz", imu_t, "#eb6834"),
        ("wheel odom  20 Hz", odo_t, "#1baf7a"), ("wifi  28 scans", None, "#eda100"),
        ("lidar → GT  ~8.6 Hz", gt_t, "#e87ba4")]

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.6), height_ratios=[2.1, 1],
                             constrained_layout=True)
Z0, Z1 = 60.0, 61.2
for ax, lw_ in ((a1, 0.25), (a2, 1.2)):
    for i, (name, tt, c) in enumerate(rows):
        y = len(rows) - 1 - i
        if tt is None:
            ax.broken_barh(list(zip(w0, w1 - w0)), (y - 0.3, 0.6), color=c, alpha=0.55, lw=0)
        else:
            ax.eventplot(tt, lineoffsets=y, linelengths=0.6, colors=c, lw=lw_)
    ax.set_yticks(range(len(rows)), [r[0] for r in rows][::-1], fontsize=9)
    ax.set_ylim(-0.55, len(rows) - 0.45)
a1.set_ylim(-1.0, len(rows) - 0.45)
for (ta, tb, lab) in ((-2.6, tA, "stationary"), (tB, 118, "stationary")):
    a1.axvspan(ta, tb, color="0.55", alpha=0.18, lw=0)
    a1.text((max(ta, -2.6) + min(tb, 118)) / 2, -0.62, lab, ha="center", va="top",
            fontsize=8, color="0.35")
a1.axvspan(Z0, Z1, color="gold", alpha=0.35, lw=0)
a1.set_xlim(-2.6, 118)
a1.set_title("Five sensor streams, four independent clocks — raw asynchronous timeline")
a2.set_xlim(Z0, Z1)
a2.set_xlabel("time since first camera frame [s]")
a2.set_title("zoom: 1.2 s while driving — only imu & wheel odom share a tick; "
             "the rest never align", fontsize=9)
plt.show()

# 2 · Camera

In [ ]:
# Eight frames evenly spaced along the path, each located on a mini-map
from PIL import Image
CAM = "#2a78d6"
d = np.hypot(np.diff(ds.camera.x), np.diff(ds.camera.y))
s = np.concatenate([[0.0], np.cumsum(d)])
idx = np.searchsorted(s, np.linspace(0, s[-1], 9)[:-1])

fig, axes = plt.subplots(2, 4, figsize=(13, 5.4))
for ax, k in zip(axes.ravel(), idx):
    ax.imshow(Image.open(f"{DATA}/camera/images/{ds.camera.filename.iloc[k]}"))
    ax.set_title(f"t = {cam_t[k]:.0f} s", fontsize=9)
    ax.axis("off")
    ia = ax.inset_axes([0.66, 0.63, 0.32, 0.35])
    ia.patch.set_alpha(0.85)
    ia.plot(ds.gt.x, ds.gt.y, c="#c3c2b7", lw=1)
    ia.plot(ds.camera.x.iloc[k], ds.camera.y.iloc[k], "o", c=CAM, ms=4)
    ia.set_aspect("equal"); ia.set_xticks([]); ia.set_yticks([])
    for sp in ia.spines.values():
        sp.set_visible(True); sp.set_edgecolor("#c3c2b7")
fig.suptitle("What the camera saw — eight places around the lap", y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
# Sweep all 3465 frames at 1/4 scale (grayscale) -> health traces + extreme frames
import scipy.ndimage as ndi
G = np.empty((len(ds.camera), 154, 205), np.uint8)
for i, fn in enumerate(ds.camera.filename):
    im = Image.open(f"{DATA}/camera/images/{fn}")
    im.draft("L", (205, 154))
    G[i] = np.asarray(im.convert("L").resize((205, 154)))
bright = G.mean((1, 2))
sharp = np.array([ndi.laplace(g.astype(np.float32)).var() for g in G])
frame = lambda k: Image.open(f"{DATA}/camera/images/{ds.camera.filename.iloc[k]}")

picks = [("brightest", np.argmax(bright)), ("darkest", np.argmin(bright)),
         ("sharpest", np.argmax(sharp)), ("blurriest", np.argmin(sharp))]
fig = plt.figure(figsize=(11, 6.6), constrained_layout=True)
gs = fig.add_gridspec(3, 4, height_ratios=[1.5, 1, 1])
b1 = fig.add_subplot(gs[1, :]); b2 = fig.add_subplot(gs[2, :], sharex=b1)
for j, (lab, k) in enumerate(picks):
    ta = fig.add_subplot(gs[0, j])
    ta.imshow(frame(k)); ta.axis("off")
    ta.set_title(f"{'abcd'[j]} · {lab}  t = {cam_t[k]:.0f} s", fontsize=8)
    tr, val = (b1, bright[k]) if j < 2 else (b2, sharp[k])
    tr.plot(cam_t[k], val, "o", ms=5, c="#0b0b0b", zorder=5)
    tr.annotate("abcd"[j], (cam_t[k], val), xytext=(0, 7),
                textcoords="offset points", ha="center", fontsize=8, color="#0b0b0b")
b1.plot(cam_t, bright, c=CAM, lw=1.0); b1.set_ylabel("brightness\n(mean gray)")
b2.plot(cam_t, sharp, c=CAM, lw=1.0); b2.set_ylabel("sharpness\n(Laplacian var)")
b2.set_xlabel("time [s]")
for ax in (b1, b2):
    ax.grid(True, axis="y")
    ax.axvspan(cam_t[0], tA, color="0.55", alpha=0.15, lw=0)
    ax.axvspan(tB, cam_t[-1], color="0.55", alpha=0.15, lw=0)
b1.set_title("Frame health over the lap — extreme frames above; gray bands = stationary",
             fontsize=10)
plt.show()

In [ ]:
# The camera as a motion sensor: consecutive-frame difference + spike frames
mad = np.abs(np.diff(G.astype(np.int16), axis=0)).mean((1, 2))
floor = np.median(mad[cam_t[1:] < tA])

def spaced_peaks(t, y, k, min_sep):
    picks = []
    for i in np.argsort(y)[::-1]:
        if all(abs(t[i] - t[p]) > min_sep for p in picks):
            picks.append(i)
            if len(picks) == k:
                break
    return sorted(picks)

pk = spaced_peaks(cam_t[1:], mad, 5, min_sep=8.0)
fig = plt.figure(figsize=(11, 5.8), constrained_layout=True)
gs = fig.add_gridspec(2, 5, height_ratios=[1.5, 1.7])
ax = fig.add_subplot(gs[1, :])
for j, i in enumerate(pk):
    ta = fig.add_subplot(gs[0, j])
    ta.imshow(frame(i + 1)); ta.axis("off")
    ta.set_title(f"{'abcde'[j]} · t = {cam_t[i + 1]:.0f} s", fontsize=8)
    ax.plot(cam_t[1:][i], mad[i], "o", ms=5, c="#0b0b0b", zorder=5)
    ax.annotate("abcde"[j], (cam_t[1:][i], mad[i]), xytext=(0, 7),
                textcoords="offset points", ha="center", fontsize=8, color="#0b0b0b")
ax.plot(cam_t[1:], mad, c=CAM, lw=1.0)
ax.grid(True, axis="y")
ax.axhline(floor, c="#898781", lw=0.8, ls="--")
ax.text(cam_t[-1], floor, "noise floor (stationary) ", fontsize=8, color="#52514e",
        va="bottom", ha="right")
ax.axvspan(cam_t[0], tA, color="0.55", alpha=0.15, lw=0)
ax.axvspan(tB, cam_t[-1], color="0.55", alpha=0.15, lw=0)
ax.set_xlabel("time [s]"); ax.set_ylabel("frame-to-frame change\n(mean |Δgray|)")
ax.set_title("The camera as a motion sensor — frames at the five biggest well-separated spikes",
             fontsize=10)
plt.show()